In [1]:

import os
import subprocess

java_home = (
    "/opt/homebrew/Cellar/openjdk@11/"
    "11.0.32.1/libexec/openjdk.jdk/Contents/Home"
)

os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = (
    f"{java_home}/bin:"
    + os.environ["PATH"]
)

result = subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True
)

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print(result.stderr)


JAVA_HOME: /opt/homebrew/Cellar/openjdk@11/11.0.32.1/libexec/openjdk.jdk/Contents/Home
openjdk version "11.0.32.1" 2026-08-18
OpenJDK Runtime Environment Homebrew (build 11.0.32.1+0)
OpenJDK 64-Bit Server VM Homebrew (build 11.0.32.1+0, mixed mode)



In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("OlistMarketingSilverDelta")
    .master("local[*]")

    # Delta Lake
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )

    # Hive Metastore
    .config(
        "spark.hadoop.hive.metastore.uris",
        "thrift://localhost:9083"
    )

    # HDFS
    .config(
        "spark.hadoop.fs.defaultFS",
        "hdfs://localhost:9000"
    )
    .config(
        "spark.hadoop.dfs.client.use.datanode.hostname",
        "true"
    )

    .enableHiveSupport()
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark:", spark.version)

print(
    "Delta extensions:",
    spark.conf.get("spark.sql.extensions")
)

print(
    "Delta catalog:",
    spark.conf.get("spark.sql.catalog.spark_catalog")
)

print(
    "Hive Metastore:",
    spark.sparkContext._jsc.hadoopConfiguration()
    .get("hive.metastore.uris")
)

26/11/11 16:43:12 WARN Utils: Your hostname, MacBook-Pro-de-Bruno.local resolves to a loopback address: 127.0.0.1; using 192.168.1.225 instead (on interface en0)
26/11/11 16:43:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/brunosousa/.ivy2/cache
The jars for the packages stored in: /Users/brunosousa/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1380b6e0-12e3-4246-b983-0c832e77d771;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 64ms :: artifacts dl 2ms
	:: modules in use:
	io.delta#delta-core_2.12;2.4.0 from central in [default]
	io.delta#delta-storage;2.4.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	-------------------------------------------------------------

:: loading settings :: url = jar:file:/Users/brunosousa/Documents/EDSTD/big-data-open-os/.venv-delta/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


26/11/11 16:43:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark: 3.4.4
Delta extensions: io.delta.sql.DeltaSparkSessionExtension
Delta catalog: org.apache.spark.sql.delta.catalog.DeltaCatalog
Hive Metastore: thrift://localhost:9083


In [3]:
spark.sql("SHOW DATABASES").show(truncate=False)

+---------+
|namespace|
+---------+
|default  |
|demo     |
+---------+



In [4]:
import os
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Olist_Marketing_Funnel_Silver")
    .master("local[*]")
    .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9000")
    .config("spark.hadoop.dfs.client.use.datanode.hostname", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

conf = spark.sparkContext._jsc.hadoopConfiguration()

print("Spark:", spark.version)
print("HDFS:", conf.get("fs.defaultFS"))
print(
    "Usar hostname do DataNode:",
    conf.get("dfs.client.use.datanode.hostname")
)

Spark: 3.4.4
HDFS: hdfs://localhost:9000
Usar hostname do DataNode: true


26/11/11 16:43:15 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [5]:

# Caminho da camada Bronze no HDFS
bronze_path = "hdfs://localhost:9000/demo/bronze/olist_marketing"

# Ficheiros de origem
ficheiros = {
    "MQL": "olist_marketing_qualified_leads_dataset.csv",
    "Closed Deals": "olist_closed_deals_dataset.csv",
    "IPCA": "IBGE_IPCA.csv"
}

# Verificar acesso aos ficheiros no HDFS
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
    spark._jvm.java.net.URI(bronze_path),
    hadoop_conf
)

for nome, ficheiro in ficheiros.items():
    caminho = f"{bronze_path}/{ficheiro}"
    existe = fs.exists(
        spark._jvm.org.apache.hadoop.fs.Path(caminho)
    )
    print(f"{nome}: {'OK' if existe else 'NÃO ENCONTRADO'}")


MQL: OK
Closed Deals: OK
IPCA: OK


In [6]:

# Consultar a localização dos blocos do primeiro CSV no HDFS

hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
    spark._jvm.java.net.URI("hdfs://localhost:9000"),
    hadoop_conf
)

path = spark._jvm.org.apache.hadoop.fs.Path(
    "/demo/bronze/olist_marketing/"
    "olist_marketing_qualified_leads_dataset.csv"
)

status = fs.getFileStatus(path)
blocos = fs.getFileBlockLocations(status, 0, status.getLen())

for bloco in blocos:
    print("Hosts:", list(bloco.getHosts()))
    print("Names:", list(bloco.getNames()))


Hosts: ['hdfs-dn1']
Names: ['172.18.0.3:9866']


In [7]:

# Teste isolado de leitura do primeiro CSV
# Sem inferência automática dos tipos

caminho_mql = (
    "hdfs://localhost:9000/demo/bronze/olist_marketing/"
    "olist_marketing_qualified_leads_dataset.csv"
)

print("A iniciar leitura...")

df_mql = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(caminho_mql)
)

print("Leitura inicial concluída.")
print("Colunas:", df_mql.columns)

df_mql.limit(5).show(truncate=False)


A iniciar leitura...
Leitura inicial concluída.
Colunas: ['mql_id', 'first_contact_date', 'landing_page_id', 'origin']
+--------------------------------+------------------+--------------------------------+--------------+
|mql_id                          |first_contact_date|landing_page_id                 |origin        |
+--------------------------------+------------------+--------------------------------+--------------+
|dac32acd4db4c29c230538b72f8dd87d|2018-02-01        |88740e65d5d6b056e0cda098e1ea6313|social        |
|8c18d1de7f67e60dbd64e3c07d7e9d5d|2017-10-20        |007f9098284a86ee80ddeb25d53e0af8|paid_search   |
|b4bc852d233dfefc5131f593b538befa|2018-03-22        |a7982125ff7aa3b2054c6e44f9d28522|organic_search|
|6be030b81c75970747525b843c1ef4f8|2018-01-22        |d45d558f0daeecf3cccdffe3c59684aa|email         |
|5420aad7fec3549a85876ba1c529bd84|2018-02-21        |b48ec5f3b04e9068441002a19df93c6c|organic_search|
+--------------------------------+------------------+------------

In [8]:
caminho = (
    "hdfs://localhost:9000/demo/bronze/olist_marketing/"
    "olist_marketing_qualified_leads_dataset.csv"
)

df_teste = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(caminho)
)

df_teste.show(5, truncate=False)

+--------------------------------+------------------+--------------------------------+--------------+
|mql_id                          |first_contact_date|landing_page_id                 |origin        |
+--------------------------------+------------------+--------------------------------+--------------+
|dac32acd4db4c29c230538b72f8dd87d|2018-02-01        |88740e65d5d6b056e0cda098e1ea6313|social        |
|8c18d1de7f67e60dbd64e3c07d7e9d5d|2017-10-20        |007f9098284a86ee80ddeb25d53e0af8|paid_search   |
|b4bc852d233dfefc5131f593b538befa|2018-03-22        |a7982125ff7aa3b2054c6e44f9d28522|organic_search|
|6be030b81c75970747525b843c1ef4f8|2018-01-22        |d45d558f0daeecf3cccdffe3c59684aa|email         |
|5420aad7fec3549a85876ba1c529bd84|2018-02-21        |b48ec5f3b04e9068441002a19df93c6c|organic_search|
+--------------------------------+------------------+--------------------------------+--------------+
only showing top 5 rows



In [9]:

# Procurar os ficheiros CSV no HDFS

hdfs_base = "hdfs://localhost:9000"

fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
    spark._jvm.java.net.URI(hdfs_base),
    hadoop_conf
)

PathHDFS = spark._jvm.org.apache.hadoop.fs.Path

def listar_hdfs(caminho, nivel=0, max_nivel=5):
    if nivel > max_nivel:
        return

    path = PathHDFS(caminho)

    if not fs.exists(path):
        print("Caminho inexistente:", caminho)
        return

    for item in fs.listStatus(path):
        nome = item.getPath().toString()

        print("  " * nivel + nome)

        if item.isDirectory():
            listar_hdfs(
                nome,
                nivel + 1,
                max_nivel
            )

print("FICHEIROS NA BRONZE:\n")

listar_hdfs("hdfs://localhost:9000/demo")


FICHEIROS NA BRONZE:

hdfs://localhost:9000/demo/bronze
  hdfs://localhost:9000/demo/bronze/episodes.csv
  hdfs://localhost:9000/demo/bronze/episodes.json
  hdfs://localhost:9000/demo/bronze/olist_marketing
    hdfs://localhost:9000/demo/bronze/olist_marketing/IBGE_IPCA.csv
    hdfs://localhost:9000/demo/bronze/olist_marketing/olist_closed_deals_dataset.csv
    hdfs://localhost:9000/demo/bronze/olist_marketing/olist_marketing_qualified_leads_dataset.csv
  hdfs://localhost:9000/demo/bronze/weather.json
hdfs://localhost:9000/demo/episodes-out
  hdfs://localhost:9000/demo/episodes-out/_delta_log
    hdfs://localhost:9000/demo/episodes-out/_delta_log/00000000000000000000.json
    hdfs://localhost:9000/demo/episodes-out/_delta_log/00000000000000000001.json
    hdfs://localhost:9000/demo/episodes-out/_delta_log/00000000000000000002.json
  hdfs://localhost:9000/demo/episodes-out/part-00000-58a2e3e9-0a81-4610-bacd-f66e8f48cf37-c000.snappy.parquet
  hdfs://localhost:9000/demo/episodes-out/part-

In [10]:
bronze_path = "hdfs://localhost:9000/demo/bronze/olist_marketing"

datasets = {
    "MQL": "olist_marketing_qualified_leads_dataset.csv",
    "Closed Deals": "olist_closed_deals_dataset.csv",
    "IPCA": "IBGE_IPCA.csv"
}

dfs = {}

for nome, ficheiro in datasets.items():
    dfs[nome] = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(f"{bronze_path}/{ficheiro}")
    )

    print(f"\n===== {nome} =====")
    print("Linhas:", dfs[nome].count())
    print("Colunas:", len(dfs[nome].columns))
    dfs[nome].printSchema()


===== MQL =====
Linhas: 8000
Colunas: 4
root
 |-- mql_id: string (nullable = true)
 |-- first_contact_date: string (nullable = true)
 |-- landing_page_id: string (nullable = true)
 |-- origin: string (nullable = true)


===== Closed Deals =====
Linhas: 842
Colunas: 14
root
 |-- mql_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- sdr_id: string (nullable = true)
 |-- sr_id: string (nullable = true)
 |-- won_date: string (nullable = true)
 |-- business_segment: string (nullable = true)
 |-- lead_type: string (nullable = true)
 |-- lead_behaviour_profile: string (nullable = true)
 |-- has_company: string (nullable = true)
 |-- has_gtin: string (nullable = true)
 |-- average_stock: string (nullable = true)
 |-- business_type: string (nullable = true)
 |-- declared_product_catalog_size: string (nullable = true)
 |-- declared_monthly_revenue: string (nullable = true)


===== IPCA =====
Linhas: 393
Colunas: 2
root
 |-- Date: string (nullable = true)
 |-- IPCA Rate 

In [11]:
from pyspark.sql import functions as F

for nome, df in dfs.items():
    print(f"\n===== {nome} =====")
    print("Total de registos:", df.count())

    # Contagem de valores nulos ou vazios por coluna
    expressoes = [
        F.sum(
            F.when(
                F.col(c).isNull() |
                (F.trim(F.col(c)) == ""),
                1
            ).otherwise(0)
        ).alias(c)
        for c in df.columns
    ]

    df.select(*expressoes).show(truncate=False)

# Verificar duplicados nas chaves de ligação
for nome in ["MQL", "Closed Deals"]:
    df = dfs[nome]

    print(f"\nDuplicados de mql_id em {nome}:")

    (
        df.groupBy("mql_id")
        .count()
        .filter(F.col("count") > 1)
        .show(10, truncate=False)
    )

# Inspecionar os valores e formatos do IPCA
print("\n===== AMOSTRA IPCA =====")
dfs["IPCA"].show(15, truncate=False)


===== MQL =====
Total de registos: 8000
+------+------------------+---------------+------+
|mql_id|first_contact_date|landing_page_id|origin|
+------+------------------+---------------+------+
|0     |0                 |0              |60    |
+------+------------------+---------------+------+


===== Closed Deals =====
Total de registos: 842
+------+---------+------+-----+--------+----------------+---------+----------------------+-----------+--------+-------------+-------------+-----------------------------+------------------------+
|mql_id|seller_id|sdr_id|sr_id|won_date|business_segment|lead_type|lead_behaviour_profile|has_company|has_gtin|average_stock|business_type|declared_product_catalog_size|declared_monthly_revenue|
+------+---------+------+-----+--------+----------------+---------+----------------------+-----------+--------+-------------+-------------+-----------------------------+------------------------+
|0     |0        |0     |0    |0       |1               |6        |17

In [12]:
from pyspark.sql import functions as F

# 1. Verificar duplicados
for nome in ["MQL", "Closed Deals"]:
    df = dfs[nome]

    duplicados = (
        df.groupBy("mql_id")
        .count()
        .filter(F.col("count") > 1)
    )

    print(f"{nome} - IDs duplicados:", duplicados.count())

# 2. Verificar intervalos temporais
print("\nDatas MQL:")
dfs["MQL"].select(
    F.min("first_contact_date").alias("min"),
    F.max("first_contact_date").alias("max")
).show()

print("\nDatas Closed Deals:")
dfs["Closed Deals"].select(
    F.min("won_date").alias("min"),
    F.max("won_date").alias("max")
).show()

# 3. Converter datas do IPCA para identificar o intervalo
ipca_datas = dfs["IPCA"].withColumn(
    "data_mes",
    F.to_date(F.concat(F.lit("01/"), F.col("Date")), "dd/MM/yyyy")
)

print("\nPeríodo IPCA:")
ipca_datas.select(
    F.min("data_mes").alias("min"),
    F.max("data_mes").alias("max"),
    F.sum(F.col("data_mes").isNull().cast("int")).alias("datas_invalidas")
).show()

# 4. Verificar correspondência das chaves
mql_ids = dfs["MQL"].select("mql_id").distinct()
deals_ids = dfs["Closed Deals"].select("mql_id").distinct()

print(
    "\nClosed Deals sem MQL correspondente:",
    deals_ids.join(mql_ids, "mql_id", "left_anti").count()
)

MQL - IDs duplicados: 0
Closed Deals - IDs duplicados: 0

Datas MQL:
+----------+----------+
|       min|       max|
+----------+----------+
|2017-06-14|2018-05-31|
+----------+----------+


Datas Closed Deals:
+-------------------+-------------------+
|                min|                max|
+-------------------+-------------------+
|2017-12-05 02:00:00|2018-11-14 18:04:19|
+-------------------+-------------------+


Período IPCA:
+----------+----------+---------------+
|       min|       max|datas_invalidas|
+----------+----------+---------------+
|1994-01-01|2026-09-01|              0|
+----------+----------+---------------+


Closed Deals sem MQL correspondente: 0


In [13]:
from pyspark.sql import functions as F

# 1. SILVER MQL
silver_mql = (
    dfs["MQL"]
    .withColumn("mql_id", F.trim(F.col("mql_id")))
    .withColumn(
        "first_contact_date",
        F.to_date("first_contact_date", "yyyy-MM-dd")
    )
    .withColumn(
    "origin",
    F.when(
        F.col("origin").isNull() |
        (F.trim(F.col("origin")) == ""),
        F.lit("unknown")
    ).otherwise(
        F.lower(F.trim(F.col("origin")))
    )
    )
    .withColumn("contact_month", F.trunc("first_contact_date", "month"))
)

# 2. SILVER CLOSED DEALS
silver_deals = (
    dfs["Closed Deals"]
    .withColumn("mql_id", F.trim(F.col("mql_id")))
    .withColumn("seller_id", F.trim(F.col("seller_id")))
    .withColumn(
        "won_date",
        F.to_timestamp("won_date", "yyyy-MM-dd HH:mm:ss")
    )
    .withColumn(
        "declared_monthly_revenue",
        F.col("declared_monthly_revenue").cast("double")
    )
    .withColumn(
        "declared_product_catalog_size",
        F.col("declared_product_catalog_size").cast("integer")
    )
    .withColumn(
        "average_stock",
        F.when(
            F.col("average_stock").isNull() |
            (F.trim(F.col("average_stock")) == "") |
            (F.lower(F.trim(F.col("average_stock"))) == "unknown"),
            F.lit(None).cast("string")
            ).otherwise(
        F.trim(F.col("average_stock"))
        )
    )
)

# 3. SILVER IPCA
silver_ipca = (
    dfs["IPCA"]
    .withColumn(
        "reference_month",
        F.to_date(
            F.concat(F.lit("01/"), F.col("Date")),
            "dd/MM/yyyy"
        )
    )
    .withColumn(
        "ipca_rate_12m",
        F.col("IPCA Rate (last 12 months)").cast("double")
    )
    .select("reference_month", "ipca_rate_12m")
)

print("Silver MQL:", silver_mql.count())
print("Silver Closed Deals:", silver_deals.count())
print("Silver IPCA:", silver_ipca.count())

silver_mql.printSchema()
silver_deals.printSchema()
silver_ipca.printSchema()

Silver MQL: 8000
Silver Closed Deals: 842
Silver IPCA: 393
root
 |-- mql_id: string (nullable = true)
 |-- first_contact_date: date (nullable = true)
 |-- landing_page_id: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- contact_month: date (nullable = true)

root
 |-- mql_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- sdr_id: string (nullable = true)
 |-- sr_id: string (nullable = true)
 |-- won_date: timestamp (nullable = true)
 |-- business_segment: string (nullable = true)
 |-- lead_type: string (nullable = true)
 |-- lead_behaviour_profile: string (nullable = true)
 |-- has_company: string (nullable = true)
 |-- has_gtin: string (nullable = true)
 |-- average_stock: string (nullable = true)
 |-- business_type: string (nullable = true)
 |-- declared_product_catalog_size: integer (nullable = true)
 |-- declared_monthly_revenue: double (nullable = true)

root
 |-- reference_month: date (nullable = true)
 |-- ipca_rate_12m: double (nullab

In [14]:
from pyspark.sql import functions as F

# 1. Verificar valores nulos após conversão
for nome, df, colunas in [
    ("MQL", silver_mql, ["first_contact_date", "contact_month"]),
    ("Closed Deals", silver_deals, [
        "won_date",
        "average_stock",
        "declared_product_catalog_size",
        "declared_monthly_revenue"
    ]),
    ("IPCA", silver_ipca, ["reference_month", "ipca_rate_12m"])
]:
    print(f"\n===== {nome} =====")

    df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in colunas
    ]).show()

# 2. Verificar os valores reais do IPCA
print("\n===== AMOSTRA IPCA =====")
silver_ipca.orderBy("reference_month").show(10, truncate=False)

# 3. Verificar unicidade dos meses do IPCA
print("\n===== DUPLICADOS IPCA =====")
silver_ipca.groupBy("reference_month") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()


===== MQL =====
+------------------+-------------+
|first_contact_date|contact_month|
+------------------+-------------+
|                 0|            0|
+------------------+-------------+


===== Closed Deals =====
+--------+-------------+-----------------------------+------------------------+
|won_date|average_stock|declared_product_catalog_size|declared_monthly_revenue|
+--------+-------------+-----------------------------+------------------------+
|       0|          780|                          773|                       0|
+--------+-------------+-----------------------------+------------------------+


===== IPCA =====
+---------------+-------------+
|reference_month|ipca_rate_12m|
+---------------+-------------+
|              0|            0|
+---------------+-------------+


===== AMOSTRA IPCA =====
+---------------+-------------+
|reference_month|ipca_rate_12m|
+---------------+-------------+
|1994-01-01     |2693.84      |
|1994-02-01     |3035.71      |
|1994-03-01    

In [15]:
from pyspark.sql import functions as F

# Dataset original da camada Bronze
df_deals = dfs["Closed Deals"]

print("===== VALORES ORIGINAIS DE AVERAGE_STOCK =====")

df_deals.groupBy("average_stock") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(30, truncate=False)

print("\n===== VALORES NÃO NULOS NA BRONZE =====")

df_deals.filter(
    F.col("average_stock").isNotNull()
).select(
    "average_stock"
).show(20, truncate=False)

===== VALORES ORIGINAIS DE AVERAGE_STOCK =====
+-------------+-----+
|average_stock|count|
+-------------+-----+
|null         |776  |
|5-20         |22   |
|50-200       |15   |
|1-5          |10   |
|20-50        |8    |
|200+         |7    |
|unknown      |4    |
+-------------+-----+


===== VALORES NÃO NULOS NA BRONZE =====
+-------------+
|average_stock|
+-------------+
|20-50        |
|20-50        |
|1-5          |
|1-5          |
|5-20         |
|5-20         |
|5-20         |
|200+         |
|1-5          |
|1-5          |
|50-200       |
|5-20         |
|50-200       |
|200+         |
|200+         |
|200+         |
|5-20         |
|50-200       |
|1-5          |
|50-200       |
+-------------+
only showing top 20 rows



In [16]:
from pyspark.sql import functions as F

# Identificar os negócios fechados
deals_join = silver_deals.withColumn(
    "converted",
    F.lit(1)
)

# Integrar MQL com Closed Deals
silver_integrated = (
    silver_mql.alias("m")
    .join(
        deals_join.alias("d"),
        on="mql_id",
        how="left"
    )
    .withColumn(
        "converted",
        F.coalesce(F.col("converted"), F.lit(0))
    )
)

# Associar o IPCA ao mês do primeiro contacto
silver_integrated = (
    silver_integrated.alias("base")
    .join(
        silver_ipca.alias("i"),
        F.col("base.contact_month") == F.col("i.reference_month"),
        "left"
    )
    .drop("reference_month")
)

print("Registos da tabela integrada:", silver_integrated.count())
print("Negócios convertidos:",
      silver_integrated.filter(F.col("converted") == 1).count())

silver_integrated.select(
    "mql_id",
    "first_contact_date",
    "origin",
    "converted",
    "won_date",
    "business_segment",
    "ipca_rate_12m"
).show(10, truncate=False)

Registos da tabela integrada: 8000
Negócios convertidos: 842
+--------------------------------+------------------+--------------+---------+-------------------+----------------+-------------+
|mql_id                          |first_contact_date|origin        |converted|won_date           |business_segment|ipca_rate_12m|
+--------------------------------+------------------+--------------+---------+-------------------+----------------+-------------+
|dac32acd4db4c29c230538b72f8dd87d|2018-02-01        |social        |0        |null               |null            |2.84         |
|8c18d1de7f67e60dbd64e3c07d7e9d5d|2017-10-20        |paid_search   |0        |null               |null            |2.7          |
|b4bc852d233dfefc5131f593b538befa|2018-03-22        |organic_search|0        |null               |null            |2.68         |
|6be030b81c75970747525b843c1ef4f8|2018-01-22        |email         |0        |null               |null            |2.86         |
|5420aad7fec3549a85876ba1c529

In [17]:
print("===== VALIDAÇÃO SILVER INTEGRADA =====")

total = silver_integrated.count()

convertidos = silver_integrated.filter(
    F.col("converted") == 1
).count()

sem_ipca = silver_integrated.filter(
    F.col("ipca_rate_12m").isNull()
).count()

duplicados = (
    silver_integrated.groupBy("mql_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Total de registos:", total)
print("Convertidos:", convertidos)
print("Não convertidos:", total - convertidos)
print("Registos sem IPCA:", sem_ipca)
print("MQL IDs duplicados:", duplicados)

print(
    "Taxa de conversão:",
    round(convertidos / total * 100, 2),
    "%"
)

===== VALIDAÇÃO SILVER INTEGRADA =====
Total de registos: 8000
Convertidos: 842
Não convertidos: 7158
Registos sem IPCA: 0
MQL IDs duplicados: 0
Taxa de conversão: 10.53 %


In [18]:
silver_path = "hdfs://localhost:9000/demo/silver/olist_marketing_integrated"

(
    silver_integrated.write
    .mode("overwrite")
    .parquet(silver_path)
)

print("Tabela Silver guardada em:", silver_path)

Tabela Silver guardada em: hdfs://localhost:9000/demo/silver/olist_marketing_integrated


In [19]:
silver_check = spark.read.parquet(silver_path)

print("Registos recuperados:", silver_check.count())
print("Colunas:", len(silver_check.columns))

silver_check.select(
    "mql_id",
    "origin",
    "converted",
    "contact_month",
    "ipca_rate_12m"
).show(5, truncate=False)

Registos recuperados: 8000
Colunas: 20
+--------------------------------+--------------+---------+-------------+-------------+
|mql_id                          |origin        |converted|contact_month|ipca_rate_12m|
+--------------------------------+--------------+---------+-------------+-------------+
|dac32acd4db4c29c230538b72f8dd87d|social        |0        |2018-02-01   |2.84         |
|8c18d1de7f67e60dbd64e3c07d7e9d5d|paid_search   |0        |2017-10-01   |2.7          |
|b4bc852d233dfefc5131f593b538befa|organic_search|0        |2018-03-01   |2.68         |
|6be030b81c75970747525b843c1ef4f8|email         |0        |2018-01-01   |2.86         |
|5420aad7fec3549a85876ba1c529bd84|organic_search|1        |2018-02-01   |2.84         |
+--------------------------------+--------------+---------+-------------+-------------+
only showing top 5 rows



In [20]:
print("Catálogo Spark:")
print(spark.conf.get("spark.sql.catalogImplementation"))

print("\nHive Metastore URI:")
print(
    spark.sparkContext._jsc.hadoopConfiguration()
    .get("hive.metastore.uris")
)

print("\nBases de dados disponíveis:")
spark.sql("SHOW DATABASES").show(truncate=False)

Catálogo Spark:
hive

Hive Metastore URI:
thrift://localhost:9083

Bases de dados disponíveis:
+---------+
|namespace|
+---------+
|default  |
|demo     |
+---------+



In [21]:
print("Spark:", spark.version)
print("Warehouse:", spark.conf.get("spark.sql.warehouse.dir"))
print("Catálogo:", spark.conf.get("spark.sql.catalogImplementation"))

print("\nConfigurações Hive:")
for chave, valor in spark.sparkContext.getConf().getAll():
    if "hive" in chave.lower() or "metastore" in chave.lower():
        print(chave, "=", valor)

Spark: 3.4.4
Warehouse: file:/Users/brunosousa/Documents/EDSTD/big-data-open-os/Semana%204/spark-warehouse
Catálogo: hive

Configurações Hive:
spark.hadoop.hive.metastore.uris = thrift://localhost:9083


In [22]:
silver_path = "hdfs://localhost:9000/demo/silver/olist_marketing_integrated"

# Confirmar que o Parquet continua acessível
silver_check = spark.read.parquet(silver_path)

print("Registos:", silver_check.count())
print("Colunas:", len(silver_check.columns))

# Registar a tabela no catálogo Hive
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS demo.silver_marketing_integrated
    USING PARQUET
    LOCATION '{silver_path}'
""")

print("Tabela registada no Hive Metastore.")

Registos: 8000
Colunas: 20
Tabela registada no Hive Metastore.


In [23]:
spark.sql("SHOW TABLES IN demo").show(truncate=False)

spark.sql("""
    SELECT
        COUNT(*) AS total_leads,
        SUM(CASE WHEN converted = 1 THEN 1 ELSE 0 END) AS converted_leads,
        ROUND(AVG(CAST(converted AS DOUBLE)) * 100, 2) AS conversion_rate
    FROM demo.silver_marketing_integrated
""").show()

+---------+---------------------------+-----------+
|namespace|tableName                  |isTemporary|
+---------+---------------------------+-----------+
|demo     |parquet_table              |false      |
|demo     |silver_marketing_integrated|false      |
|demo     |text_table                 |false      |
+---------+---------------------------+-----------+

+-----------+---------------+---------------+
|total_leads|converted_leads|conversion_rate|
+-----------+---------------+---------------+
|       8000|            842|          10.53|
+-----------+---------------+---------------+



In [24]:
spark.sql("""
    DESCRIBE EXTENDED demo.silver_marketing_integrated
""").filter(
    "col_name IN ('Type', 'Provider', 'Location')"
).show(truncate=False)

+--------+------------------------------------------------------------+-------+
|col_name|data_type                                                   |comment|
+--------+------------------------------------------------------------+-------+
|Type    |EXTERNAL                                                    |       |
|Provider|PARQUET                                                     |       |
|Location|hdfs://localhost:9000/demo/silver/olist_marketing_integrated|       |
+--------+------------------------------------------------------------+-------+



In [25]:
print("Spark:", spark.version)

print(
    "Delta extensions:",
    spark.conf.get(
        "spark.sql.extensions",
        "Não configurado"
    )
)

print(
    "Delta catalog:",
    spark.conf.get(
        "spark.sql.catalog.spark_catalog",
        "Não configurado"
    )
)

Spark: 3.4.4
Delta extensions: io.delta.sql.DeltaSparkSessionExtension
Delta catalog: org.apache.spark.sql.delta.catalog.DeltaCatalog


In [26]:
import importlib.metadata

print("PySpark:", importlib.metadata.version("pyspark"))

try:
    print("Delta Lake:", importlib.metadata.version("delta-spark"))
except importlib.metadata.PackageNotFoundError:
    print("Delta Lake não está instalado neste ambiente.")

PySpark: 3.4.4
Delta Lake: 2.4.0


In [27]:
print("===== BASES DE DADOS =====")
spark.sql("SHOW DATABASES").show(truncate=False)

print("===== TABELAS DA BASE DEMO =====")
spark.sql("SHOW TABLES IN demo").show(truncate=False)

print("===== VALIDAÇÃO SILVER =====")
spark.sql("""
    SELECT
        COUNT(*) AS total_registos,
        SUM(converted) AS total_convertidos
    FROM demo.silver_marketing_integrated
""").show()

===== BASES DE DADOS =====
+---------+
|namespace|
+---------+
|default  |
|demo     |
+---------+

===== TABELAS DA BASE DEMO =====
+---------+---------------------------+-----------+
|namespace|tableName                  |isTemporary|
+---------+---------------------------+-----------+
|demo     |parquet_table              |false      |
|demo     |silver_marketing_integrated|false      |
|demo     |text_table                 |false      |
+---------+---------------------------+-----------+

===== VALIDAÇÃO SILVER =====
+--------------+-----------------+
|total_registos|total_convertidos|
+--------------+-----------------+
|          8000|              842|
+--------------+-----------------+



In [28]:
# Caminho da nova tabela Delta no HDFS
delta_path = "hdfs://localhost:9000/demo/silver/olist_marketing_integrated_delta"

# Ler a Silver existente
silver_df = spark.table("demo.silver_marketing_integrated")

# Escrever em formato Delta
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .save(delta_path)
)

print("Silver escrita em formato Delta com sucesso.")

Silver escrita em formato Delta com sucesso.


In [29]:
# Ler diretamente a tabela Delta
delta_check = spark.read.format("delta").load(delta_path)

print("Registos:", delta_check.count())
print("Colunas:", len(delta_check.columns))

delta_check.groupBy("converted").count().show()

Registos: 8000
Colunas: 20
+---------+-----+
|converted|count|
+---------+-----+
|        1|  842|
|        0| 7158|
+---------+-----+



In [30]:
from delta.tables import DeltaTable

print("É uma tabela Delta:", DeltaTable.isDeltaTable(spark, delta_path))

delta_table = DeltaTable.forPath(spark, delta_path)

delta_table.history().select(
    "version", "timestamp", "operation"
).show(truncate=False)

É uma tabela Delta: True
+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|0      |2026-11-11 16:45:00.897|WRITE    |
+-------+-----------------------+---------+



In [31]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS demo.silver_marketing_integrated_delta
    USING DELTA
    LOCATION '{delta_path}'
""")

print("Tabela Delta registada no Hive Metastore.")

Tabela Delta registada no Hive Metastore.


In [32]:
spark.sql("SHOW TABLES IN demo").show(truncate=False)

spark.sql("""
    SELECT
        COUNT(*) AS total_leads,
        SUM(CASE WHEN converted = 1 THEN 1 ELSE 0 END) AS converted_leads,
        ROUND(AVG(CAST(converted AS DOUBLE)) * 100, 2) AS conversion_rate
    FROM demo.silver_marketing_integrated_delta
""").show()

+---------+---------------------------------+-----------+
|namespace|tableName                        |isTemporary|
+---------+---------------------------------+-----------+
|demo     |parquet_table                    |false      |
|demo     |silver_marketing_integrated      |false      |
|demo     |silver_marketing_integrated_delta|false      |
|demo     |text_table                       |false      |
+---------+---------------------------------+-----------+

+-----------+---------------+---------------+
|total_leads|converted_leads|conversion_rate|
+-----------+---------------+---------------+
|       8000|            842|          10.53|
+-----------+---------------+---------------+

